# Integrated Clean — Passive BBO Maker + Exact-0.01 Sniper

Combines `maker_clean` and `peak_clean` into one strategy:

- **No 0.01 exposed at the effective top of book** → run the maker: be the best bid AND the
  best ask (improve-first, join-fallback, never take on arrival).
- **Exactly 0.01 BTC exposed at the effective top** → snipe it: take that order with a
  marketable limit at its exact price (0.01 at the ask → BUY it, price expected to rise;
  0.01 at the bid → SELL into it, price expected to fall).

## The "effective top" trick
The snipe signal is evaluated on the book **with our own resting quotes netted out**. That
one expression covers every case, including the tricky one:

| Raw book | Netted book | Meaning | Action |
|---|---|---|---|
| 0.01 is the BBO outright | 0.01 is effective L1 | classic peak signal | snipe |
| **our quote alone at BBO, exactly 0.01 directly behind it** | our level vanishes → the 0.01 is effective L1 | we're first in line in front of the trap | **confirm-cancel our own quote, then take the 0.01 immediately** |
| our quote at BBO but others joined our level | residual size ≠ 0.01 | we're NOT alone — can't be sure what's behind | no signal, keep quoting |
| we joined a level that itself shows 0.01 residual after netting us out | 0.01 is effective L1 | we're resting next to the trap | cancel ours, take the rest |

"Am I the only one at my level?" falls out automatically: netting subtracts exactly our
0.0004 from our posted level — if anything remains, someone else is there and the signal
doesn't fire.

## Interaction rules (priority: never-self-cross > snipe > BBO presence)
1. **Snipe is checked first on every websocket event**, before any maker logic — the
   detection → order path is in-memory checks, an optional confirm-cancel of our own
   crossing quote, and one POST.
2. **Never cross our own order**: if our resting quote sits at or inside the snipe price,
   it is cancel-confirmed first; if the cancel can't be confirmed, the snipe is aborted.
3. **Maker pauses while a take is in flight** (no new quotes until the taker order resolves,
   ≤ `TAKE_TIMEOUT_SECS`); resting quotes are left alone.
4. **Maker stands down against the thesis**: no new bid while a 0.01 is exposed on the bid
   side (we expect price to fall through it), no new ask while one is exposed on the ask
   side — even when the snipe itself is blocked by the inventory band. We never build
   inventory in front of a level we believe is about to get run over.
5. **Inventory band ±0.001 BTC hard**, enforced on both paths: maker clips are 0.0004,
   snipes are 0.001. A signal that would breach the band is skipped.

## Safety & rate limit (carried from both parents)
- Maker side: cross guard, freshness gate, crossed-book gate, CB-mid sanity, last-instant
  recheck in `place_passive_limit`, self-cross guard, cancel-confirm, ghost detection,
  reprice cooldowns, hold-the-BBO (queue priority is sacred).
- Sniper side: marketable limit at the exposed price only (never a market order, never
  chases), one-shot arming per exposure, entry cooldown, timeout-cancel, last-instant
  netted re-validation in `place_taker_limit`.
- Budget: fills usually confirm from the create response (1 request per snipe, +1 when our
  own quote must be cancelled first); balance poll every 4s doubles as maker-fill detection
  and inventory reconciliation; requests reserve so a cancel is always affordable.
- $5 session loss limit; every exit path cancels all orders.

## Caveats
- The DEPTH feed aggregates by price level — "exactly 0.01" may be several orders summing
  to 0.01, and "alone at our level" is inferred from displayed size, not order identity.
- A partial fill on a timed-out taker order is trued up by the balance poll (it will log as
  an unattributed fill and confirm-kill resting quotes as a precaution, like maker_clean).

In [ ]:
import sys
import os
import asyncio
import math
import time
import json
from pathlib import Path

ROOT_DIR = Path(os.getcwd()).parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import aiohttp
import websockets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from dotenv import load_dotenv

from lib.execution import ExecutionClient
from lib.coinbase_feed import CoinbaseBookA

load_dotenv(ROOT_DIR / 'keys' / '.env')

TM_REST_URL = os.getenv('BASE_REST_URL', 'https://api.truemarkets.co')
TM_KEY_FILE = str(ROOT_DIR / 'keys' / 'truemarkets-api-key-edd1691b.json')
CB_WS_URL   = os.getenv('COINBASE_WS_URL', 'wss://ws-feed.exchange.coinbase.com')
CB_PRODUCT  = 'BTC-USD'
BASE_ASSET  = 'BTC'
QUOTE_ASSET = 'USDC'


In [ ]:
class TrueMarketsBook:
    _WS_URL = 'wss://api.truex.co/api/v1'
    _HEADERS = {
        'Origin': 'https://truemarkets.co',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    def __init__(self, symbol='BTC-PYUSD'):
        self.symbol    = symbol
        self._bids     = {}
        self._asks     = {}
        self._ready    = False
        self._last_msg = None            # wall-clock time of last websocket message
        self.updated   = asyncio.Event() # set on every book change — lets the strategy react instantly

    @property
    def best_bid(self):      return max(self._bids) if self._bids else None
    @property
    def best_ask(self):      return min(self._asks) if self._asks else None
    @property
    def best_bid_size(self):
        b = self.best_bid; return self._bids[b] if b is not None else None
    @property
    def best_ask_size(self):
        a = self.best_ask; return self._asks[a] if a is not None else None
    @property
    def mid(self):
        b, a = self.best_bid, self.best_ask
        return (b + a) / 2.0 if (b is not None and a is not None) else None
    @property
    def is_ready(self): return self._ready
    @property
    def age(self):
        """Seconds since the last websocket message, or None if never connected."""
        return (time.time() - self._last_msg) if self._last_msg else None

    def _handle_snapshot(self, data):
        self._bids  = {float(b['price']): float(b['qty']) for b in data.get('bids', []) if float(b['qty']) > 0}
        self._asks  = {float(a['price']): float(a['qty']) for a in data.get('asks', []) if float(a['qty']) > 0}
        self._ready = True
        self.updated.set()

    def _handle_update(self, data):
        for b in data.get('bids', []):
            p, q = float(b['price']), float(b['qty'])
            if q == 0: self._bids.pop(p, None)
            else:      self._bids[p] = q
        for a in data.get('asks', []):
            p, q = float(a['price']), float(a['qty'])
            if q == 0: self._asks.pop(p, None)
            else:      self._asks[p] = q
        self.updated.set()

    async def run(self):
        backoff = 1
        while True:
            try:
                async with websockets.connect(self._WS_URL, additional_headers=self._HEADERS) as ws:
                    backoff = 1
                    self._ready = False
                    await ws.send(json.dumps({
                        'type': 'SUBSCRIBE_NO_AUTH',
                        'item_names': [self.symbol],
                        'channels': ['DEPTH'],
                        'timestamp': str(int(time.time())),
                    }))
                    async for raw in ws:
                        self._last_msg = time.time()
                        try: msg = json.loads(raw)
                        except Exception: continue
                        t = msg.get('update')
                        d = msg.get('data', {})
                        if   t == 'SNAPSHOT': self._handle_snapshot(d)
                        elif t == 'UPDATE':   self._handle_update(d)
            except websockets.ConnectionClosed:
                pass
            except Exception as e:
                print(f'TrueMarketsBook error: {e}')
            self._ready = False
            self.updated.set()   # wake the strategy so it notices the outage immediately
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

### Parameters

In [ ]:
# ── sizes ────────────────────────────────────────────────────────────────────
QUOTE_SIZE_BTC     = 0.0004  # EVERY maker quote, both sides, is exactly this size
TAKE_SIZE_BTC      = 0.001   # EVERY snipe is exactly this size
TRIGGER_SIZE_BTC   = 0.01    # snipe only when the effective BBO shows EXACTLY this size
SIZE_EPS           = 1e-9    # float tolerance for the exact-size comparison
TICK_SIZE          = 0.1
PRICE_DECIMALS     = 1
IDLE_TICK_SECS     = 1.0     # heartbeat when the websocket is quiet (housekeeping only)

# ── risk limits ──────────────────────────────────────────────────────────────
MAX_POSITION_BTC   = 0.001   # hard inventory band, both paths: never net beyond this
DAILY_LOSS_LIMIT   = 5.0     # session kill switch ($)

# ── maker: never-take safety gates (from maker_clean) ────────────────────────
CROSS_GUARD_TICKS  = 1       # min full ticks between our quote and the live opposite touch
PULL_QUOTES_AFTER_SECS = 30.0  # feed dead this long -> cancel resting quotes entirely
REPRICE_MIN_SECS   = 1.0     # per-side cancel/replace cooldown (rate-limit protection)
ORDER_VERIFY_SECS  = 60.0    # re-verify long-lived resting orders against the exchange
GHOST_GRACE_SECS   = 3.0     # let the feed catch up after placing before "missing" checks
GHOST_CHECK_MIN_SECS = 5.0   # min gap between missing-from-book status checks

# ── sniper: entry pacing (from peak_clean) ───────────────────────────────────
TAKE_COOLDOWN_SECS = 2.0     # min gap between snipe attempts
TAKE_TIMEOUT_SECS  = 3.0     # unfilled taker limit is cancelled after this — never chase
FILL_CHECK_SECS    = 0.8     # min gap between status polls on an in-flight taker order

# ── shared gates ─────────────────────────────────────────────────────────────
MAX_BOOK_AGE_SECS  = 5.0     # book with no tick in this long is untrusted: no new orders
MAX_CB_DEVIATION   = 0.01    # TM mid vs Coinbase mid sanity bound (1%)
RL_RESERVE         = 8       # keep this many requests in reserve so a cancel is always possible
BALANCE_POLL_SECS  = 4.0     # balance poll = maker-fill detection + inventory reconciliation

PLOT_EVERY_SECS    = 5.0
STATUS_EVERY_SECS  = 1.0
TM_BOOK_SYMBOL     = 'BTC-PYUSD'

# order statuses that mean "still live on the exchange"
LIVE_STATUSES = {'pending', 'active', 'open', 'new', 'partially_filled'}

### Helpers

In [ ]:
quote_size_str = f'{QUOTE_SIZE_BTC:.8f}'.rstrip('0').rstrip('.')
take_size_str  = f'{TAKE_SIZE_BTC:.8f}'.rstrip('0').rstrip('.')
assert quote_size_str == '0.0004', f'quote size must be exactly 0.0004, got {quote_size_str}'
assert take_size_str  == '0.001',  f'take size must be exactly 0.001, got {take_size_str}'

def fmt_px(p): return f'{p:.{PRICE_DECIMALS}f}'

def floor_tick(p): return math.floor(p / TICK_SIZE + 1e-9) * TICK_SIZE
def ceil_tick(p):  return math.ceil (p / TICK_SIZE - 1e-9) * TICK_SIZE

def is_trigger(sz):
    """True when a displayed size is EXACTLY the 0.01 trigger."""
    return sz is not None and abs(sz - TRIGGER_SIZE_BTC) < SIZE_EPS

def levels_ex_own(levels, own_px=None, own_sz=0.0, reverse=False):
    """Book side as sorted (price, size) with our own resting order netted out.

    This is what the market looks like without us: if we are alone at our level
    the level disappears entirely; if others joined it, only their residual
    remains. The snipe signal and its last-instant re-validation both run on
    this view, so 'our quote is BBO with exactly 0.01 directly behind it' and
    '0.01 is the BBO outright' are the same condition."""
    out = []
    for p, q in levels.items():
        if own_px is not None and abs(p - own_px) < 1e-9:
            q -= own_sz
            if q < SIZE_EPS:
                continue
        out.append((p, q))
    out.sort(reverse=reverse)
    return out


async def cancel_confirmed(bot, session, oid):
    """True only when the order is CONFIRMED no longer live.

    A swallowed cancel failure must never free a slot — that is how duplicate
    orders (and self-crosses) happen. If the cancel is not acknowledged, fall
    back to reading the order status; anything unknown counts as live."""
    if not oid:
        return True
    try:
        if await bot.cancel_order(session, oid):
            return True
    except Exception:
        pass
    try:
        st = await bot.get_order_status(session, oid)
    except Exception:
        return False
    if st is None:
        return False
    return st not in LIVE_STATUSES


async def place_passive_limit(bot, session, tm_book, side, px):
    """The ONLY passive-order path (maker quotes). Re-validates everything
    against the live book at the last possible instant and refuses anything
    that could take. Returns (order, refusal_reason) — exactly one is set."""
    if abs(QUOTE_SIZE_BTC - 0.0004) > 1e-12:
        return None, 'size drifted from 0.0004'
    if abs(px - round(px / TICK_SIZE) * TICK_SIZE) > 1e-6:
        return None, f'price {px} not tick-aligned'
    px_str = fmt_px(px)
    if abs(float(px_str) - px) > 1e-6:
        return None, f'price string {px_str} != {px}'
    if not tm_book.is_ready or tm_book.age is None or tm_book.age > MAX_BOOK_AGE_SECS:
        return None, 'book stale at send time'
    lb, la = tm_book.best_bid, tm_book.best_ask
    if lb is None or la is None or lb >= la:
        return None, 'book empty/crossed at send time'
    guard = CROSS_GUARD_TICKS * TICK_SIZE
    if side == 'buy':
        if px > la - guard + 1e-9:
            return None, f'bid {px_str} within {CROSS_GUARD_TICKS} tick(s) of ask {la:.1f}'
    elif side == 'sell':
        if px < lb + guard - 1e-9:
            return None, f'ask {px_str} within {CROSS_GUARD_TICKS} tick(s) of bid {lb:.1f}'
    else:
        return None, f'bad side {side!r}'
    order = await bot.place_order(
        session, BASE_ASSET, QUOTE_ASSET,
        side, quote_size_str, 'base', 'limit', px_str
    )
    return order, None


async def place_taker_limit(bot, session, tm_book, side, px, own_px=None):
    """The ONLY aggressive-order path (snipes). A marketable limit at the
    exposed 0.01's exact price: it takes instantly if the trigger is still
    there, and by construction can never fill at a worse price than the one we
    saw. `own_px` nets our own (possibly just-cancelled, possibly still shown
    by the lagging feed) quote out of the snapshot so validation sees what the
    market will see once we are gone. Refuses rather than rests or chases.

    Returns (order, refusal_reason) — exactly one of the two is set."""
    if abs(TAKE_SIZE_BTC - 0.001) > 1e-12:
        return None, 'size drifted from 0.001'
    if abs(px - round(px / TICK_SIZE) * TICK_SIZE) > 1e-6:
        return None, f'price {px} not tick-aligned'
    px_str = fmt_px(px)
    if abs(float(px_str) - px) > 1e-6:
        return None, f'price string {px_str} != {px}'
    if not tm_book.is_ready or tm_book.age is None or tm_book.age > MAX_BOOK_AGE_SECS:
        return None, 'book stale at send time'
    if side == 'buy':
        lvls = levels_ex_own(tm_book._asks, own_px, QUOTE_SIZE_BTC)
        if not lvls:
            return None, 'no asks at send time'
        bpx, bsz = lvls[0]
        if abs(px - bpx) > 1e-9:
            return None, f'effective ask moved to {bpx:.1f} != {px_str}'
        if not is_trigger(bsz):
            return None, 'trigger size gone from ask'
    elif side == 'sell':
        lvls = levels_ex_own(tm_book._bids, own_px, QUOTE_SIZE_BTC, reverse=True)
        if not lvls:
            return None, 'no bids at send time'
        bpx, bsz = lvls[0]
        if abs(px - bpx) > 1e-9:
            return None, f'effective bid moved to {bpx:.1f} != {px_str}'
        if not is_trigger(bsz):
            return None, 'trigger size gone from bid'
    else:
        return None, f'bad side {side!r}'
    order = await bot.place_order(
        session, BASE_ASSET, QUOTE_ASSET,
        side, take_size_str, 'base', 'limit', px_str
    )
    return order, None


async def get_full_balances(bot, session):
    data = await bot._get(session, '/v1/conductor/balances')
    total_btc = avail_btc = total_usd = avail_usd = 0.0
    if not data:
        return None, None, None, None
    for b in data.get('data', []):
        sym   = b.get('symbol', '')
        avail = float(b.get('available', 0) or 0)
        held  = float(b.get('held',      0) or 0)
        if sym == BASE_ASSET:
            total_btc += avail + held
            avail_btc += avail
        elif sym in (QUOTE_ASSET, 'PYUSD'):
            total_usd += avail + held
            avail_usd += avail
    return total_btc, total_usd, avail_btc, avail_usd

# ── chart series ─────────────────────────────────────────────────────────────
time_series = []
pnl_series  = []
mvol_series = []
tvol_series = []
inv_series  = []
trades      = []   # (elapsed_secs, side, px) — snipes only

def _cumsum(vals):
    out, s = [], 0.0
    for v in vals:
        s += v; out.append(s)
    return out

def update_plot(tm_book=None, our_bid=None, our_ask=None, last_take_px=None, last_take_side=None):
    clear_output(wait=True)
    fig, axes = plt.subplots(4, 1, figsize=(12, 16))
    ax1, ax2, ax3, ax4 = axes

    ax1.plot(time_series, pnl_series, color='green', label='Session PnL ($)')
    ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax1.axhline(-DAILY_LOSS_LIMIT, color='red', linestyle='--', linewidth=1,
                label=f'Loss limit -${DAILY_LOSS_LIMIT}')
    ax1.fill_between(time_series, pnl_series, 0,
                     where=[p >= 0 for p in pnl_series], alpha=0.15, color='green')
    ax1.fill_between(time_series, pnl_series, 0,
                     where=[p <  0 for p in pnl_series], alpha=0.15, color='red')
    ax1.set_ylabel('PnL ($)'); ax1.set_title('Session PnL')
    ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)

    ax2.plot(time_series, mvol_series, color='blue',   label='Maker volume ($)')
    ax2.plot(time_series, tvol_series, color='purple', label='Taker volume ($)')
    ax2.fill_between(time_series, mvol_series, alpha=0.10, color='blue')
    ax2.fill_between(time_series, tvol_series, alpha=0.10, color='purple')
    ax2.set_ylabel('Volume ($)'); ax2.set_title('Cumulative Volume — Maker vs Taker')
    ax2.legend(loc='upper left'); ax2.grid(True, alpha=0.3)

    ax3.plot(time_series, inv_series, color='orange', label='Net inventory (BTC)')
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.axhline( MAX_POSITION_BTC, color='red', linestyle=':', alpha=0.6, label='Inventory band')
    ax3.axhline(-MAX_POSITION_BTC, color='red', linestyle=':', alpha=0.6)
    buys  = [t for t, s, _ in trades if s == 'buy']
    sells = [t for t, s, _ in trades if s == 'sell']
    if buys:
        ax3.scatter(buys,  [TAKE_SIZE_BTC]  * len(buys),  marker='^',
                    color='green', s=60, zorder=5, label='Buy snipes')
    if sells:
        ax3.scatter(sells, [-TAKE_SIZE_BTC] * len(sells), marker='v',
                    color='red',   s=60, zorder=5, label='Sell snipes')
    ax3.set_ylabel('BTC'); ax3.set_title(f'Net Inventory — {len(trades)} snipes')
    ax3.legend(loc='upper left'); ax3.grid(True, alpha=0.3)

    if tm_book is not None and tm_book.is_ready and tm_book._bids and tm_book._asks:
        sorted_bids = sorted(tm_book._bids.items(), reverse=True)
        sorted_asks = sorted(tm_book._asks.items())
        bid_px  = [p for p, _ in sorted_bids]
        bid_cum = _cumsum([q for _, q in sorted_bids])
        ask_px  = [p for p, _ in sorted_asks]
        ask_cum = _cumsum([q for _, q in sorted_asks])
        ax4.fill_between(bid_px, bid_cum, step='post', color='green', alpha=0.35, label='Bids')
        ax4.plot(bid_px, bid_cum, color='lime',   drawstyle='steps-post', linewidth=1)
        ax4.fill_between(ask_px, ask_cum, step='pre',  color='red',   alpha=0.35, label='Asks')
        ax4.plot(ask_px, ask_cum, color='salmon', drawstyle='steps-pre',  linewidth=1)
        if our_bid is not None:
            ax4.axvline(our_bid, color='lime',   linestyle='--', linewidth=1.5,
                        label=f'Our bid ${our_bid:.1f}')
        if our_ask is not None:
            ax4.axvline(our_ask, color='salmon', linestyle='--', linewidth=1.5,
                        label=f'Our ask ${our_ask:.1f}')
        if last_take_px is not None:
            c = 'lime' if last_take_side == 'buy' else 'salmon'
            ax4.axvline(last_take_px, color=c, linestyle=':', linewidth=2,
                        label=f'Last snipe {last_take_side} ${last_take_px:.1f}')
        mid = (bid_px[0] + ask_px[0]) / 2
        ax4.set_xlim(mid - 500, mid + 500)
    else:
        ax4.text(0.5, 0.5, 'Waiting for order book...', ha='center', va='center',
                 transform=ax4.transAxes)
    ax4.set_xlabel('Price ($)'); ax4.set_ylabel('Cumulative BTC')
    ax4.set_title('Live TrueMarkets Order Book Depth')
    ax4.legend(loc='upper right'); ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    display(fig)
    plt.close(fig)

### Strategy Loop

Evaluated on every websocket event, in strict order:

1. **Freshness / gates** — stale or dead feed pulls everything; crossed book, CB-mid
   deviation and rate-budget gates block all new orders.
2. **Snipe fast path** — net our quotes out of the book; if the effective BBO on either
   side is exactly 0.01 (and the side is armed): confirm-cancel our own crossing quote if
   needed, then fire one marketable limit at the exposed price. Transient blocks
   (gate/cooldown) keep the signal armed; attempts and persistent blocks (band, funds)
   disarm it until the exposure changes.
3. **In-flight take management** — throttled status polls, hard cancel at timeout.
4. **Balance poll (4s)** — detects maker fills (balance delta minus snipes booked since the
   last poll), frees the filled side, and reconciles inventory authoritatively.
5. **Ghost detection** — unchanged from maker_clean.
6. **Maker quoting** — unchanged from maker_clean (improve-first, join-fallback, hold the
   BBO), plus two new stand-downs: paused entirely while a take is in flight, and per-side
   when a 0.01 is exposed on that side (never build inventory in front of the trap).

In [ ]:
async def run_strategy():
    for s in [time_series, pnl_series, mvol_series, tvol_series, inv_series]:
        s.clear()
    trades.clear()

    bot     = ExecutionClient(key_file=TM_KEY_FILE, base_url=TM_REST_URL)
    book_a  = CoinbaseBookA(ws_url=CB_WS_URL, product_id=CB_PRODUCT)
    tm_book = TrueMarketsBook(symbol=TM_BOOK_SYMBOL)

    feed_cb = asyncio.create_task(book_a.run())
    feed_tm = asyncio.create_task(tm_book.run())

    start_time = time.time()

    # maker state (per side)
    bid_oid = ask_oid = None
    bid_posted_px = ask_posted_px = None
    bid_placed_at = ask_placed_at = None
    bid_checked_at = ask_checked_at = 0.0
    bid_repriced_at = ask_repriced_at = 0.0

    # sniper state
    pending_take      = None   # the single in-flight taker order (dict) or None
    last_buy_fired_px = None   # one-shot arming per exposure
    last_sell_fired_px = None
    last_take_at      = 0.0
    takes_since_poll  = 0.0    # signed BTC booked from snipes since the last balance poll
    last_take_px = last_take_side = None

    # accounting
    initial_btc = initial_usd = None
    prev_total_btc = None
    total_btc = total_usd = avail_btc = avail_usd = 0.0
    inventory_btc = cash_usdc = 0.0
    maker_volume = taker_volume = 0.0

    last_balance_at = last_plot_at = last_print_at = 0.0

    def record_take_fill(side, px):
        nonlocal inventory_btc, cash_usdc, taker_volume, takes_since_poll
        nonlocal avail_btc, avail_usd, last_take_px, last_take_side
        signed = TAKE_SIZE_BTC if side == 'buy' else -TAKE_SIZE_BTC
        inventory_btc    += signed
        takes_since_poll += signed
        cash_usdc        -= signed * px
        if side == 'buy':
            avail_usd -= TAKE_SIZE_BTC * px
            avail_btc += TAKE_SIZE_BTC
        else:
            avail_btc -= TAKE_SIZE_BTC
            avail_usd += TAKE_SIZE_BTC * px
        taker_volume += TAKE_SIZE_BTC * px
        last_take_px, last_take_side = px, side
        trades.append((time.time() - start_time, side, px))
        print(f'  ✓ SNIPE {side.upper()} {take_size_str} @ ${px:.1f}  inv {inventory_btc:+.5f}')

    try:
        async with aiohttp.ClientSession() as session:

            async def resolve_pending_take(try_cancel):
                """Check the in-flight taker order (cancelling first when asked).
                Books the fill if it completed. True once the order is resolved.
                A cancel can race a fill — status after the cancel is what counts."""
                nonlocal pending_take
                if not pending_take:
                    return True
                if try_cancel:
                    try:
                        await bot.cancel_order(session, pending_take['oid'])
                    except Exception:
                        pass
                st = await bot.get_order_status(session, pending_take['oid'])
                if st == 'complete':
                    record_take_fill(pending_take['side'], pending_take['px'])
                    pending_take = None
                    return True
                if st is not None and st not in LIVE_STATUSES:
                    print(f'  ! taker order ended {st} without a booked fill')
                    pending_take = None
                    return True
                return False

            await bot.authenticate(session)
            print('Authenticated. Cancelling any pre-existing orders...')
            await bot.cancel_all(session)

            res = await get_full_balances(bot, session)
            if res[0] is None:
                raise RuntimeError('Failed to fetch initial balances')
            initial_btc, initial_usd, avail_btc, avail_usd = res
            prev_total_btc = initial_btc
            total_btc, total_usd = initial_btc, initial_usd
            print(f'Starting balances: {initial_btc:.6f} BTC  |  ${initial_usd:.2f} USDC+PYUSD')
            print('Waiting for TrueMarkets book snapshot...')

            while not tm_book.is_ready:
                await asyncio.sleep(0.2)
            print(f'TM book ready: {tm_book.best_bid:.1f} / {tm_book.best_ask:.1f}')
            print(f'Quoting BBO both sides; sniping effective-BBO levels of exactly '
                  f'{TRIGGER_SIZE_BTC} BTC. Interrupt kernel to stop.\n')

            while True:
                # ── 0. Wake on the next book event (or heartbeat) ─────────────
                try:
                    await asyncio.wait_for(tm_book.updated.wait(), timeout=IDLE_TICK_SECS)
                except asyncio.TimeoutError:
                    pass
                tm_book.updated.clear()
                now = time.time()

                # ── 1. Book freshness — never act on a stale/dead book ────────
                age   = tm_book.age
                fresh = tm_book.is_ready and age is not None and age <= MAX_BOOK_AGE_SECS
                if not fresh:
                    if pending_take:
                        print(f"  {time.strftime('%H:%M:%S')}  feed stale with take in flight — pulling it")
                        await resolve_pending_take(try_cancel=True)
                    dead = (age is None) or (age > PULL_QUOTES_AFTER_SECS) or not tm_book.is_ready
                    if (bid_oid or ask_oid) and dead:
                        print(f"  {time.strftime('%H:%M:%S')}  feed dead — pulling quotes")
                        if bid_oid and await cancel_confirmed(bot, session, bid_oid):
                            bid_oid = bid_posted_px = bid_placed_at = None
                        if ask_oid and await cancel_confirmed(bot, session, ask_oid):
                            ask_oid = ask_posted_px = ask_placed_at = None
                    await asyncio.sleep(IDLE_TICK_SECS)
                    continue

                best_bid = tm_book.best_bid
                best_ask = tm_book.best_ask
                if best_bid is None or best_ask is None:
                    await asyncio.sleep(IDLE_TICK_SECS)
                    continue
                spread = best_ask - best_bid
                bid_sz = tm_book.best_bid_size or 0.0
                ask_sz = tm_book.best_ask_size or 0.0

                # ── 2. Shared gates (all in-memory) ───────────────────────────
                cb_mid = book_a.mid
                tm_mid = tm_book.mid
                rl     = bot._rl_remaining

                crossed   = best_bid >= best_ask
                deviated  = bool(cb_mid and tm_mid and abs(tm_mid - cb_mid) / cb_mid > MAX_CB_DEVIATION)
                budget_ok = (rl is None) or (rl > RL_RESERVE)

                quote_gate = None
                if crossed:
                    quote_gate = f'book crossed {best_bid:.1f}/{best_ask:.1f}'
                elif deviated:
                    quote_gate = f'TM mid {tm_mid:.1f} vs CB mid {cb_mid:.1f} > {MAX_CB_DEVIATION:.1%}'
                elif not budget_ok:
                    quote_gate = f'rate budget {rl} <= reserve {RL_RESERVE}'

                # a deviating book means our resting quotes may be badly mispriced — get out
                if deviated and (bid_oid or ask_oid):
                    print(f'  ! {quote_gate} — pulling quotes')
                    if bid_oid and await cancel_confirmed(bot, session, bid_oid):
                        bid_oid = bid_posted_px = bid_placed_at = None
                    if ask_oid and await cancel_confirmed(bot, session, ask_oid):
                        ask_oid = ask_posted_px = ask_placed_at = None

                mark = cb_mid or tm_mid

                # ── 3. SNIPE FAST PATH — the book with our own quotes netted out
                ask_lvls = levels_ex_own(tm_book._asks,
                                         ask_posted_px if ask_oid else None, QUOTE_SIZE_BTC)
                bid_lvls = levels_ex_own(tm_book._bids,
                                         bid_posted_px if bid_oid else None, QUOTE_SIZE_BTC,
                                         reverse=True)
                buy_tgt  = ask_lvls[0][0] if (ask_lvls and is_trigger(ask_lvls[0][1])) else None
                sell_tgt = bid_lvls[0][0] if (bid_lvls and is_trigger(bid_lvls[0][1])) else None

                # re-arm a side when its exposure vanishes or moves to a new price
                if last_buy_fired_px is not None and (
                        buy_tgt is None or abs(buy_tgt - last_buy_fired_px) > 1e-9):
                    last_buy_fired_px = None
                if last_sell_fired_px is not None and (
                        sell_tgt is None or abs(sell_tgt - last_sell_fired_px) > 1e-9):
                    last_sell_fired_px = None

                buy_sig  = buy_tgt  is not None and last_buy_fired_px  is None
                sell_sig = sell_tgt is not None and last_sell_fired_px is None

                snipe_side = None
                if pending_take is None:
                    if buy_sig and sell_sig:
                        snipe_side = 'sell' if inventory_btc > 0 else 'buy'  # reduce inventory
                    elif buy_sig:
                        snipe_side = 'buy'
                    elif sell_sig:
                        snipe_side = 'sell'

                snipe_action = None
                snipe_attempted = False
                if snipe_side:
                    tgt = buy_tgt if snipe_side == 'buy' else sell_tgt
                    # transient blocks keep the signal armed; attempts and
                    # persistent blocks disarm it until the exposure changes
                    if quote_gate:
                        snipe_action = f'{snipe_side} snipe @ {tgt:.1f} waiting ({quote_gate})'
                    elif now - last_take_at < TAKE_COOLDOWN_SECS:
                        snipe_action = f'{snipe_side} snipe @ {tgt:.1f} waiting (cooldown)'
                    elif snipe_side == 'buy' and inventory_btc + TAKE_SIZE_BTC > MAX_POSITION_BTC + 1e-12:
                        last_buy_fired_px = tgt
                        snipe_action = f'buy snipe @ {tgt:.1f} skipped (max long {inventory_btc:+.5f})'
                    elif snipe_side == 'sell' and inventory_btc - TAKE_SIZE_BTC < -MAX_POSITION_BTC - 1e-12:
                        last_sell_fired_px = tgt
                        snipe_action = f'sell snipe @ {tgt:.1f} skipped (max short {inventory_btc:+.5f})'
                    elif snipe_side == 'buy' and avail_usd < TAKE_SIZE_BTC * tgt * 1.01:
                        last_buy_fired_px = tgt
                        snipe_action = f'buy snipe @ {tgt:.1f} skipped (no USDC)'
                    elif snipe_side == 'sell' and avail_btc < TAKE_SIZE_BTC:
                        last_sell_fired_px = tgt
                        snipe_action = f'sell snipe @ {tgt:.1f} skipped (no BTC)'
                    else:
                        snipe_attempted = True
                        last_take_at = now
                        if snipe_side == 'buy':
                            last_buy_fired_px = tgt
                        else:
                            last_sell_fired_px = tgt

                        # never cross our own quote: confirm-kill it first if it's
                        # at or inside the snipe price — abort if unconfirmed
                        aborted = False
                        own_px = None
                        if (snipe_side == 'buy' and ask_oid and ask_posted_px is not None
                                and ask_posted_px <= tgt + 1e-9):
                            if await cancel_confirmed(bot, session, ask_oid):
                                own_px = ask_posted_px  # feed may still display it — net it out
                                ask_oid = ask_posted_px = ask_placed_at = None
                            else:
                                aborted = True
                                snipe_action = 'buy snipe aborted (own ask cancel unconfirmed)'
                        elif (snipe_side == 'sell' and bid_oid and bid_posted_px is not None
                                and bid_posted_px >= tgt - 1e-9):
                            if await cancel_confirmed(bot, session, bid_oid):
                                own_px = bid_posted_px
                                bid_oid = bid_posted_px = bid_placed_at = None
                            else:
                                aborted = True
                                snipe_action = 'sell snipe aborted (own bid cancel unconfirmed)'

                        if not aborted:
                            order, refusal = await place_taker_limit(
                                bot, session, tm_book, snipe_side, tgt, own_px=own_px)
                            if order and order.get('order_id'):
                                st = order.get('status')
                                if st == 'complete':
                                    record_take_fill(snipe_side, tgt)
                                    snipe_action = f'SNIPE {snipe_side.upper()} filled on create @ {tgt:.1f}'
                                else:
                                    pending_take = {'oid': order['order_id'], 'side': snipe_side,
                                                    'px': tgt, 'placed_at': now, 'checked_at': now}
                                    snipe_action = f'SNIPE {snipe_side.upper()} {take_size_str} @ {tgt:.1f} sent ({st})'
                            elif refusal:
                                snipe_action = f'snipe refused ({refusal})'
                            else:
                                snipe_action = f'SNIPE {snipe_side.upper()} FAILED'

                # ── 4. In-flight take management (throttled polls, hard timeout)
                if pending_take and now - pending_take['checked_at'] >= FILL_CHECK_SECS:
                    pending_take['checked_at'] = now
                    timed_out = now - pending_take['placed_at'] >= TAKE_TIMEOUT_SECS
                    if timed_out:
                        print(f'  ! taker order unfilled after {TAKE_TIMEOUT_SECS:.0f}s — cancelling, not chasing')
                    await resolve_pending_take(try_cancel=timed_out)

                # ── 5. Balances: maker-fill detection + inventory reconciliation
                if now - last_balance_at >= BALANCE_POLL_SECS:
                    last_balance_at = now
                    res = await get_full_balances(bot, session)
                    if res[0] is not None:
                        total_btc, total_usd, avail_btc, avail_usd = res
                        # subtract snipes we already booked locally since the last
                        # poll — the remainder is maker-fill (or unattributed) flow
                        delta_btc = total_btc - prev_total_btc - takes_since_poll
                        takes_since_poll = 0.0
                        if abs(delta_btc) > 1e-8:
                            if delta_btc > 0:
                                fill_px = bid_posted_px or best_bid
                                print(f'  ✓ MAKER BID FILL +{delta_btc:.6f} BTC @ ~${fill_px:.1f}')
                            else:
                                fill_px = ask_posted_px or best_ask
                                print(f'  ✓ MAKER ASK FILL {delta_btc:.6f} BTC @ ~${fill_px:.1f}')
                            maker_volume += abs(delta_btc) * fill_px
                            # exactly one clean 0.0004 fill -> free that side only;
                            # anything else (partials / overlaps / stray take
                            # residue) -> confirm-kill both sides so no half-filled
                            # ghost order can linger
                            if abs(delta_btc - QUOTE_SIZE_BTC) < 1e-6:
                                bid_oid = bid_posted_px = bid_placed_at = None
                            elif abs(delta_btc + QUOTE_SIZE_BTC) < 1e-6:
                                ask_oid = ask_posted_px = ask_placed_at = None
                            else:
                                if bid_oid and await cancel_confirmed(bot, session, bid_oid):
                                    bid_oid = bid_posted_px = bid_placed_at = None
                                if ask_oid and await cancel_confirmed(bot, session, ask_oid):
                                    ask_oid = ask_posted_px = ask_placed_at = None
                        inventory_btc  = total_btc - initial_btc
                        cash_usdc      = total_usd - initial_usd
                        prev_total_btc = total_btc

                # ── 6. Ghost-order detection (maker quotes) ───────────────────
                if bid_oid and bid_posted_px is not None:
                    missing = best_bid < bid_posted_px - 1e-9
                    settled = bid_placed_at and (now - bid_placed_at) > GHOST_GRACE_SECS
                    overdue = now - max(bid_placed_at or 0.0, bid_checked_at) > ORDER_VERIFY_SECS
                    if (missing and settled and now - bid_checked_at > GHOST_CHECK_MIN_SECS) or overdue:
                        st = await bot.get_order_status(session, bid_oid)
                        bid_checked_at = now
                        if st is not None and st not in LIVE_STATUSES:
                            print(f'  ! resting bid is {st} — slot freed, requoting')
                            bid_oid = bid_posted_px = bid_placed_at = None
                if ask_oid and ask_posted_px is not None:
                    missing = best_ask > ask_posted_px + 1e-9
                    settled = ask_placed_at and (now - ask_placed_at) > GHOST_GRACE_SECS
                    overdue = now - max(ask_placed_at or 0.0, ask_checked_at) > ORDER_VERIFY_SECS
                    if (missing and settled and now - ask_checked_at > GHOST_CHECK_MIN_SECS) or overdue:
                        st = await bot.get_order_status(session, ask_oid)
                        ask_checked_at = now
                        if st is not None and st not in LIVE_STATUSES:
                            print(f'  ! resting ask is {st} — slot freed, requoting')
                            ask_oid = ask_posted_px = ask_placed_at = None

                # ── 7. Maker quoting (paused entirely while a take is in flight)
                bid_action = ask_action = 'idle'
                our_bid = our_ask = None

                if pending_take is not None:
                    bid_action = ask_action = 'paused (take in flight)'
                else:
                    # targets: improve by one tick, join only if the guard blocks
                    guard = CROSS_GUARD_TICKS * TICK_SIZE
                    bid_cap = best_ask - guard
                    if ask_oid and ask_posted_px is not None:
                        bid_cap = min(bid_cap, ask_posted_px - TICK_SIZE)   # never cross our own ask
                    our_bid  = floor_tick(min(best_bid + TICK_SIZE, bid_cap))
                    bid_mode = ('improve' if our_bid > best_bid + 1e-9 else
                                'join'    if our_bid > best_bid - 1e-9 else 'blocked')

                    ask_floor = best_bid + guard
                    if bid_oid and bid_posted_px is not None:
                        ask_floor = max(ask_floor, bid_posted_px + TICK_SIZE)  # never cross our own bid
                    our_ask  = ceil_tick(max(best_ask - TICK_SIZE, ask_floor))
                    ask_mode = ('improve' if our_ask < best_ask - 1e-9 else
                                'join'    if our_ask < best_ask + 1e-9 else 'blocked')

                    # ── BID ───────────────────────────────────────────────────
                    if bid_oid and bid_posted_px is not None:
                        if best_bid <= bid_posted_px + 1e-9:
                            bid_action = 'hold (we are BBO)'      # never cancel the BBO — queue prio
                        elif quote_gate:
                            bid_action = f'hold ({quote_gate})'
                        elif now - bid_repriced_at < REPRICE_MIN_SECS:
                            bid_action = 'outbid — reprice cooldown'
                        elif await cancel_confirmed(bot, session, bid_oid):
                            bid_oid = bid_posted_px = bid_placed_at = None
                            bid_repriced_at = now
                            bid_action = 'outbid — repricing'
                        else:
                            bid_action = 'cancel unconfirmed — holding slot'

                    if not bid_oid:
                        if quote_gate:
                            bid_action = f'skip ({quote_gate})'
                        elif sell_tgt is not None:
                            # 0.01 exposed on the bid side: price expected to fall
                            # through it — never build a bid in front of the trap
                            bid_action = f'stand down (0.01 exposed at bid {sell_tgt:.1f})'
                        elif bid_mode == 'blocked':
                            bid_action = f'stand down (spread {spread:.1f} below guard)'
                        elif inventory_btc + QUOTE_SIZE_BTC > MAX_POSITION_BTC + 1e-12:
                            bid_action = f'skip (max long {inventory_btc:+.5f})'
                        elif avail_usd < QUOTE_SIZE_BTC * our_bid * 1.01:
                            bid_action = 'skip (no USDC)'
                        else:
                            order, refusal = await place_passive_limit(bot, session, tm_book, 'buy', our_bid)
                            if order and order.get('order_id'):
                                bid_oid       = order['order_id']
                                bid_posted_px = our_bid
                                bid_placed_at = now
                                bid_action    = f'BUY  {quote_size_str} @ {our_bid:.1f} ({bid_mode})'
                            elif refusal:
                                bid_action = f'refused ({refusal})'
                            else:
                                bid_action = 'BUY FAILED'

                    # ── ASK ───────────────────────────────────────────────────
                    if ask_oid and ask_posted_px is not None:
                        if best_ask >= ask_posted_px - 1e-9:
                            ask_action = 'hold (we are BBO)'      # never cancel the BBO — queue prio
                        elif quote_gate:
                            ask_action = f'hold ({quote_gate})'
                        elif now - ask_repriced_at < REPRICE_MIN_SECS:
                            ask_action = 'undercut — reprice cooldown'
                        elif await cancel_confirmed(bot, session, ask_oid):
                            ask_oid = ask_posted_px = ask_placed_at = None
                            ask_repriced_at = now
                            ask_action = 'undercut — repricing'
                        else:
                            ask_action = 'cancel unconfirmed — holding slot'

                    if not ask_oid:
                        if quote_gate:
                            ask_action = f'skip ({quote_gate})'
                        elif buy_tgt is not None:
                            # 0.01 exposed on the ask side: price expected to rise
                            # through it — never build an ask in front of the trap
                            ask_action = f'stand down (0.01 exposed at ask {buy_tgt:.1f})'
                        elif ask_mode == 'blocked':
                            ask_action = f'stand down (spread {spread:.1f} below guard)'
                        elif inventory_btc - QUOTE_SIZE_BTC < -MAX_POSITION_BTC - 1e-12:
                            ask_action = f'skip (max short {inventory_btc:+.5f})'
                        elif avail_btc < QUOTE_SIZE_BTC:
                            ask_action = 'skip (no BTC)'
                        else:
                            order, refusal = await place_passive_limit(bot, session, tm_book, 'sell', our_ask)
                            if order and order.get('order_id'):
                                ask_oid       = order['order_id']
                                ask_posted_px = our_ask
                                ask_placed_at = now
                                ask_action    = f'SELL {quote_size_str} @ {our_ask:.1f} ({ask_mode})'
                            elif refusal:
                                ask_action = f'refused ({refusal})'
                            else:
                                ask_action = 'SELL FAILED'

                # ── 8. Chart and status (throttled — actions are not) ─────────
                total_pnl = cash_usdc + inventory_btc * (mark or 0)
                elapsed   = time.time() - start_time

                time_series.append(elapsed)
                pnl_series.append(total_pnl)
                mvol_series.append(maker_volume)
                tvol_series.append(taker_volume)
                inv_series.append(inventory_btc)

                if now - last_plot_at >= PLOT_EVERY_SECS:
                    last_plot_at = now
                    update_plot(tm_book,
                                bid_posted_px if bid_posted_px is not None else our_bid,
                                ask_posted_px if ask_posted_px is not None else our_ask,
                                last_take_px, last_take_side)

                acted = snipe_attempted or any(k in bid_action + ask_action for k in
                            ('BUY', 'SELL', 'repricing', 'refused', 'FAILED', 'freed'))
                if acted or now - last_print_at >= STATUS_EVERY_SECS:
                    last_print_at = now
                    print(
                        f"  {time.strftime('%H:%M:%S')}  "
                        f"book {best_bid:.1f}[{bid_sz:.4f}]/{best_ask:.1f}[{ask_sz:.4f}]  "
                        f"spread ${spread:.2f}  "
                        f"inv {inventory_btc:+.5f}  pnl ${total_pnl:+.4f}  "
                        f"mvol ${maker_volume:.2f}  tvol ${taker_volume:.2f}  "
                        f"snipes {len(trades)}  rl {rl if rl is not None else '?'}/100"
                    )
                    if snipe_action:
                        print(f'    snipe: {snipe_action}')
                    if pending_take:
                        print(f"    awaiting take fill: {pending_take['side']} {take_size_str} @ {pending_take['px']:.1f}")
                    print(f'    bid: {bid_action}  |  ask: {ask_action}')

                # ── 9. Kill switch ────────────────────────────────────────────
                if total_pnl < -DAILY_LOSS_LIMIT:
                    print(f'\nLoss limit hit (${total_pnl:.4f}). Cancelling all orders.')
                    await bot.cancel_all(session)
                    break

    except (asyncio.CancelledError, KeyboardInterrupt):
        print('\nInterrupted.')
    finally:
        feed_cb.cancel()
        feed_tm.cancel()
        # never leave orders on the book, no matter how we exited
        try:
            async with aiohttp.ClientSession() as cleanup_session:
                await bot.cancel_all(cleanup_session)
            print('All orders cancelled on exit.')
        except BaseException as e:
            print(f'!! cancel_all on exit failed ({e!r}) — CHECK OPEN ORDERS MANUALLY')
        print('Stopped.')

In [ ]:
# Interrupt kernel (stop button) to exit cleanly — all orders are cancelled on the way out
await run_strategy()